In [ ]:
import pandas as pd
from repwise.gcloud import get_dataframe

In [ ]:
df = get_dataframe()
print(f"Loaded dataframe: {len(df)} rows")
display(df.head(3))

In [ ]:
df.columns = df.columns.str.lower().str.strip()

In [ ]:
notes = df.copy()

df = df.drop(columns=["notes"], errors="ignore")
print(f"After dropping Notes: {len(df)} rows")
display(df.head(3))

In [ ]:
df["date"] = pd.to_datetime(
    df["date"].replace("", pd.NA), errors="coerce", dayfirst=True
)
df["date"] = df["date"].ffill()
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

print(f"After date parsing: {len(df)} rows")
print(df[["date"]].head(5))

In [ ]:
df["record"] = df["record"].str.strip(" ,").str.split(",")
df = df.explode(column="record", ignore_index=True)
df["record"] = df["record"].str.strip()

print(f"After exploding records: {len(df)} rows")
print(df[["exercise", "record"]].head(10))

In [ ]:
missed = df[df["record"].isna()].copy()

In [ ]:
df = df.dropna()
print(f"After dropping missing rows: {len(df)} rows")
display(df.head(3))

In [ ]:
assert not df.isna().sum().sum()

---

In [ ]:
PATTERN = r"(\d+)x(\d+)@(\d+(?:\.\d+)?)"

invalid = df.loc[~df["record"].str.fullmatch(PATTERN, na=False)]

assert invalid.empty, (
    f"Found {len(invalid)} invalid Record values:\n{invalid['record'].tolist()}"
)

In [ ]:
df[["sets", "reps", "weight"]] = df["record"].str.extract(PATTERN)

# Explicitly convert
df["sets"] = df["sets"].astype(int)
df["reps"] = df["reps"].astype(int)
df["weight"] = df["weight"].astype(float)

df = df.drop(columns=["record"], errors="ignore")

df["order"] = df.groupby(["date", "exercise"]).cumcount() + 1

print(f"After parsing record fields: {len(df)} rows")
display(df.head(10))

---

In [ ]:
import sqlite3
from pyprojroot import here

db_path = here("db/repwise.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS staging")
    cur.execute(
        """
        CREATE TABLE staging (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date TEXT NOT NULL,
            exercise TEXT NOT NULL,
            sets INTEGER NOT NULL,
            reps INTEGER NOT NULL,
            weight REAL NOT NULL,
            "order" INTEGER NOT NULL
        );
        """
    )

    # Insert using explicit column names, passing only required DataFrame columns
    cols = ["date", "exercise", "sets", "reps", "weight", "order"]
    cur.executemany(
        "INSERT INTO staging ('date', 'exercise', 'sets', 'reps', 'weight', 'order') VALUES (?, ?, ?, ?, ?, ?)",
        df[cols].to_numpy(),
    )

print(f"Created database at {db_path}")

In [ ]:
with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS exercises (
            id INTEGER PRIMARY KEY NOT NULL,
            name TEXT UNIQUE NOT NULL
        )
        """
    )

    exercises = cur.execute(
        "SELECT DISTINCT exercise FROM staging ORDER BY exercise"
    ).fetchall()

    cur.executemany(
        "INSERT OR IGNORE INTO exercises (name) VALUES(?)",
        [e for e in exercises],
    )

In [ ]:
with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS sessions (
            id INTEGER PRIMARY KEY NOT NULL,
            date TEXT UNIQUE NOT NULL
        )
        """
    )

    dates = cur.execute("SELECT DISTINCT date FROM staging ORDER BY date").fetchall()

    cur.executemany(
        "INSERT OR IGNORE INTO sessions (date) VALUES(?)",
        [d for d in dates],
    )

In [ ]:
with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()

    # Enable Foreign Key support in SQLite (disabled by default)
    cur.execute("PRAGMA foreign_keys = ON;")

    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS entries (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id INTEGER NOT NULL,
            exercise_id INTEGER NOT NULL,
            sets INTEGER NOT NULL,
            reps INTEGER NOT NULL,
            weight REAL NOT NULL,
            "order" INTEGER NOT NULL,
            FOREIGN KEY (session_id) REFERENCES sessions (id) ON DELETE CASCADE,
            FOREIGN KEY (exercise_id) REFERENCES exercises (id) ON DELETE CASCADE
        );
        """
    )

    cur.execute(
        """
        INSERT INTO entries (session_id, exercise_id, sets, reps, weight, "order")
        SELECT 
            s.id AS session_id,
            e.id AS exercise_id,
            st.sets,
            st.reps,
            st.weight,
            st."order"
        FROM staging st
        JOIN sessions s ON st.date = s.date
        JOIN exercises e ON st.exercise = e.name;
        """
    )

---

In [ ]:
def schema(db_path) -> list[tuple]:
    """"""
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()

        result = cur.execute(
            "SELECT * FROM sqlite_schema WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        )

        tables = result.fetchall()

    return tables


schema(db_path)

In [ ]:
def get_schema(db_path) -> dict[str, list[dict]]:
    """Returns the schema mapping each table to its list of PRAGMA table_info dicts."""
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.cursor()

        tables = cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchall()

        return {
            row["name"]: [
                dict(col)
                for col in cur.execute(f"PRAGMA table_info('{row['name']}')").fetchall()
            ]
            for row in tables
        }


get_schema(db_path)